# 💊 NLP / NER Pipeline — Prescription Entity Extraction
### Stage: HTR Output → Medicine Name · Dosage · Frequency

---

## 📁 How This Fits Your Pipeline

```
Raw Prescription Photo
        ↓
DeepLab (medicine region detection)
        ↓
CRAFT (line segmentation → 128×32 crops)
        ↓
HTR Model (line crops → raw text)
        ↓
🔵 THIS NOTEBOOK (NLP / NER)
        ↓
{ medicine_name, dosage, frequency, confidence }
```

## 🏗️ Architecture (5 Layers)
```
Layer 1  Pre-processing          HTR noise correction
Layer 2  Bio-ClinicalBERT + CRF  Primary NER (drug / dosage / frequency)
Layer 2B GLiNER-BioMed           Fallback NER for unseen drug names
Layer 3  Rule-based parser       Regex for dosage units + frequency abbrevs
Layer 4  RapidFuzz + RxNorm      Fuzzy normalization of drug name spelling
Layer 5  Confidence gate         Score threshold + structured JSON output
```

## 📦 Output
```
htr_input/
└── nlp_output/
    ├── results.json          ← all prescriptions structured
    ├── flagged_review.json   ← low-confidence items
    └── nlp_pipeline.pkl      ← saved pipeline for production use
```

---
## ⚙️ Step 0 — Install Dependencies
Run this cell once. Restart kernel after installation.

In [4]:
# ── Install all required packages ──────────────────────────────────────────
import subprocess, sys

packages = [
    # Core NLP
    "transformers>=4.40.0",
    "torch",
    "spacy",
    # Bio-ClinicalBERT + CRF
    "torchcrf",          # CRF layer on top of BERT
    "seqeval",           # NER evaluation metrics
    # GLiNER-BioMed (2025 zero-shot fallback)
    "gliner",
    # Fuzzy matching + normalization
    "rapidfuzz",
    "requests",          # for RxNorm API calls
    # Utilities
    "pandas",
    "tqdm",
    "colorama",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Download spaCy English model
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])

print("\n✅ All packages installed. Restart the kernel, then run from Step 1.")

  DEPRECATION: Building 'seqeval' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'seqeval'. Discussion can be found at https://github.com/pypa/pip/issues/6334
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openxlab 0.1.3 requires requests~=2.28.2, but you have requests 2.33.1 which is incompatible.
openxlab 0.1.3 requires rich~=13.4.2, but you have rich 15.0.0 which is incompatible.
openxlab 0.1.3 requires tqdm~=4.65.0, but you have tqdm 4.67.3 which is incompatible.


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')

✅ All packages installed. Restart the kernel, then run from Step 1.


---
## 📁 Step 1 — Configuration & Path Setup

In [5]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # Add BEFORE importing torch
import torch

In [6]:
import os, json, re, pickle, warnings
import torch
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from colorama import Fore, Style, init
init(autoreset=True)
warnings.filterwarnings('ignore')

# ── PROJECT ROOT — Update this to match your machine ──────────────────────
PROJECT_ROOT = "/Users/avishkashenan/Desktop/Line Segmentation New"
# --------------------------------------------------------------------------

# ── Paths derived from your existing pipeline_config.json ─────────────────
HTR_INPUT_DIR  = os.path.join(PROJECT_ROOT, "htr_input")
LINE_CROPS_DIR = os.path.join(HTR_INPUT_DIR, "line_crops")
NLP_OUTPUT_DIR = os.path.join(HTR_INPUT_DIR, "nlp_output")
CONFIG_PATH    = os.path.join(HTR_INPUT_DIR, "pipeline_config.json")

# Create NLP output folder
os.makedirs(NLP_OUTPUT_DIR, exist_ok=True)

# ── Device setup (MPS for M4 Pro, CUDA, or CPU) ───────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print(f"{Fore.GREEN}✅ Using Apple MPS (M4 Pro GPU)")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"{Fore.GREEN}✅ Using CUDA GPU")
else:
    DEVICE = torch.device("cpu")
    print(f"{Fore.YELLOW}⚠️  Using CPU (slower, but works)")

# ── Load existing pipeline config ─────────────────────────────────────────
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        pipeline_config = json.load(f)
    print(f"{Fore.GREEN}✅ Loaded pipeline_config.json")
    print(f"   Line crops: {pipeline_config.get('dataset_stats', {}).get('total_line_crops', 'unknown')}")
else:
    pipeline_config = {}
    print(f"{Fore.YELLOW}⚠️  pipeline_config.json not found — continuing without it")

print(f"\n{Fore.CYAN}📁 NLP output will be saved to:")
print(f"   {NLP_OUTPUT_DIR}")

✅ Using Apple MPS (M4 Pro GPU)
✅ Loaded pipeline_config.json
   Line crops: 2954

📁 NLP output will be saved to:
   /Users/avishkashenan/Desktop/Line Segmentation New/htr_input/nlp_output


---
## 🔤 Step 2 — HTR Simulation / Input
In production your HTR model feeds text here. For now we simulate realistic noisy HTR output to build and test the full NLP pipeline.

In [7]:
# ── Simulated HTR outputs ─────────────────────────────────────────────────
# These look like realistic noisy HTR output from handwritten prescriptions.
# Replace with your actual HTR model output when available.

SAMPLE_HTR_OUTPUTS = [
    # Clean-ish examples
    {"id": "rx_001", "line": "Tab Amoxicillin 500mg 1x3"},
    {"id": "rx_002", "line": "Tab Paracetamol 500 mg TDS"},
    {"id": "rx_003", "line": "Syp Augmentin 228mg/5ml BD"},
    {"id": "rx_004", "line": "Cap Omeprazole 20mg OD"},
    # Noisy HTR examples (spelling errors, OCR artifacts)
    {"id": "rx_005", "line": "Tab Amoxl 500rng 1xd"},           # rng instead of mg, 1xd
    {"id": "rx_006", "line": "Tab Metroniclazol 400mg BD"},     # misspelled
    {"id": "rx_007", "line": "Cap 0meprazole 2Omg OD"},        # 0 vs O, 2O vs 20
    {"id": "rx_008", "line": "lbuprofen 4OOmg TDS"},           # l vs I, 4OO vs 400
    {"id": "rx_009", "line": "Tab Atorvastatln 10mg ON"},      # ln vs in
    {"id": "rx_010", "line": "Syp Amoxicilin 125mg/5ml 1x3"}, # one l missing
    # Abbreviation-heavy examples
    {"id": "rx_011", "line": "Tab PCM 500mg q8h"},
    {"id": "rx_012", "line": "Tab MTZ 400 BD x5/7"},
    {"id": "rx_013", "line": "Cap Clox 500 QID"},
    {"id": "rx_014", "line": "Inj Gentamycin 80mg IM OD"},
    {"id": "rx_015", "line": "Tab Cetirizne 10mg nocte"},      # misspelled
]

print(f"{Fore.GREEN}✅ {len(SAMPLE_HTR_OUTPUTS)} HTR lines loaded")
print(f"\n{Fore.CYAN}Sample inputs:")
for item in SAMPLE_HTR_OUTPUTS[:5]:
    print(f"  [{item['id']}] {item['line']}")
print("  ...")

# ── PRODUCTION USAGE ───────────────────────────────────────────────────────
# When your HTR model is ready, replace SAMPLE_HTR_OUTPUTS with:
#
# htr_results = []  # your HTR model output
# for img_file in sorted(os.listdir(LINE_CROPS_DIR)):
#     text = your_htr_model.predict(img_file)   # ← plug your model here
#     htr_results.append({"id": img_file, "line": text})
# SAMPLE_HTR_OUTPUTS = htr_results

✅ 15 HTR lines loaded

Sample inputs:
  [rx_001] Tab Amoxicillin 500mg 1x3
  [rx_002] Tab Paracetamol 500 mg TDS
  [rx_003] Syp Augmentin 228mg/5ml BD
  [rx_004] Cap Omeprazole 20mg OD
  [rx_005] Tab Amoxl 500rng 1xd
  ...


---
## 🧹 Step 3 — Layer 1: Pre-processing (HTR Noise Correction)
Based on Paper 9 (OCR pipeline) and the hybrid approach from Paper 6. Fixes the most common HTR artifacts before feeding to NER.

In [8]:
class PrescriptionPreprocessor:
    """
    Layer 1: Corrects common HTR/OCR noise patterns in prescription text.
    Based on: Paper 9 (OCR pipeline), Paper 6 (abbreviation expansion).
    """

    # ── Common digit/letter confusion in HTR ──────────────────────────────
    OCR_FIXES = [
    (r'(\d+)\s*rng',      r'\1mg'),
    (r'(\d+)\s*rnl',      r'\1ml'),
    (r'(\d)O(\d)',        r'\g<1>0\2'),   # mid-number O  ← also fix this one
    (r'O(\d)',            r'0\1'),
    (r'(\d)O\b',         r'\g<1>0'),      # ← THE FIX: was r'\10'
    (r'\bl([A-Z])',       r'I\1'),
    (r'([a-zA-Z])\.([a-zA-Z])', r'\1\2'),
    (r'\s+',             r' '),
]

    # ── Dosage frequency abbreviation expansion ───────────────────────────
    # Sources: standard Latin medical abbreviations
    FREQ_ABBREVS = {
        # Times per day
        r'\bOD\b':    'once daily',
        r'\bBD\b':    'twice daily',
        r'\bTDS\b':   'three times daily',
        r'\bQID\b':   'four times daily',
        r'\b1x3\b':   'three times daily',
        r'\b1xd\b':   'once daily',
        r'\b2x3\b':   'three times daily, 2 tablets',
        r'\bq8h\b':   'every 8 hours',
        r'\bq6h\b':   'every 6 hours',
        r'\bq12h\b':  'every 12 hours',
        # Timing
        r'\bON\b':    'at night',
        r'\bnocte\b': 'at night',
        r'\bMane\b':  'in the morning',
        r'\bac\b':    'before meals',
        r'\bpc\b':    'after meals',
        r'\bstat\b':  'immediately',
        # Route
        r'\bIM\b':    'intramuscular',
        r'\bIV\b':    'intravenous',
        r'\bpo\b':    'oral',
        r'\bSL\b':    'sublingual',
    }

    # ── Dosage form normalization ──────────────────────────────────────────
    FORM_ABBREVS = {
        r'\bTab\b':   'Tablet',
        r'\bCap\b':   'Capsule',
        r'\bSyp\b':   'Syrup',
        r'\bInj\b':   'Injection',
        r'\bEye\b':   'Eye',
        r'\bEar\b':   'Ear',
    }

    # ── Known abbreviation-to-generic mappings ────────────────────────────
    DRUG_ABBREVS = {
        r'\bPCM\b':   'Paracetamol',
        r'\bMTZ\b':   'Metronidazole',
        r'\bClox\b':  'Cloxacillin',
        r'\bAugmentin\b': 'Amoxicillin Clavulanate',
        r'\bASA\b':   'Aspirin',
        r'\bNSAID\b': 'NSAID',
    }

    def clean(self, text: str) -> dict:
        """Full pre-processing pipeline. Returns cleaned text + changes log."""
        original  = text
        changes   = []

        # 1. OCR artifact fixes
        for pattern, replacement in self.OCR_FIXES:
            new_text = re.sub(pattern, replacement, text)
            if new_text != text:
                changes.append(f"OCR fix: '{text}' → '{new_text}'")
                text = new_text

        # 2. Drug abbreviation expansion
        for pattern, replacement in self.DRUG_ABBREVS.items():
            new_text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
            if new_text != text:
                changes.append(f"Drug abbrev: '{pattern}' → '{replacement}'")
                text = new_text

        # 3. Form normalization
        for pattern, replacement in self.FORM_ABBREVS.items():
            new_text = re.sub(pattern, replacement, text)
            if new_text != text:
                changes.append(f"Form: '{pattern}' → '{replacement}'")
                text = new_text

        # 4. Frequency expansion (saved for later — keep in raw for rule parser)
        expanded_freq = text
        for pattern, replacement in self.FREQ_ABBREVS.items():
            expanded_freq = re.sub(pattern, replacement, expanded_freq, flags=re.IGNORECASE)

        # 5. Strip + normalize whitespace
        text          = text.strip()
        expanded_freq = expanded_freq.strip()

        return {
            "cleaned":      text,
            "expanded":     expanded_freq,   # freq abbrevs expanded — used by NER
            "original":     original,
            "changes":      changes,
            "was_modified": len(changes) > 0,
        }


# ── Test the preprocessor ─────────────────────────────────────────────────
preprocessor = PrescriptionPreprocessor()

print(f"{Fore.CYAN}{'─'*60}")
print(f"{Fore.CYAN}Layer 1: Pre-processing Results")
print(f"{Fore.CYAN}{'─'*60}")

preprocessed_lines = []
for item in SAMPLE_HTR_OUTPUTS:
    result = preprocessor.clean(item["line"])
    preprocessed_lines.append({**item, **result})

    status = f"{Fore.YELLOW}(modified)" if result["was_modified"] else f"{Fore.GREEN}(clean)"
    print(f"\n[{item['id']}] {status}")
    print(f"  Input:    {item['line']}")
    if result["was_modified"]:
        print(f"  Cleaned:  {result['cleaned']}")
        print(f"  Expanded: {result['expanded']}")
        for change in result["changes"]:
            print(f"  ✎ {change}")

print(f"\n{Fore.GREEN}✅ Pre-processing complete: {len(preprocessed_lines)} lines processed")

────────────────────────────────────────────────────────────
Layer 1: Pre-processing Results
────────────────────────────────────────────────────────────

[rx_001] (modified)
  Input:    Tab Amoxicillin 500mg 1x3
  Cleaned:  Tablet Amoxicillin 500mg 1x3
  Expanded: Tablet Amoxicillin 500mg three times daily
  ✎ Form: '\bTab\b' → 'Tablet'

[rx_002] (modified)
  Input:    Tab Paracetamol 500 mg TDS
  Cleaned:  Tablet Paracetamol 500 mg TDS
  Expanded: Tablet Paracetamol 500 mg three times daily
  ✎ Form: '\bTab\b' → 'Tablet'

[rx_003] (modified)
  Input:    Syp Augmentin 228mg/5ml BD
  Cleaned:  Syrup Amoxicillin Clavulanate 228mg/5ml BD
  Expanded: Syrup Amoxicillin Clavulanate 228mg/5ml twice daily
  ✎ Drug abbrev: '\bAugmentin\b' → 'Amoxicillin Clavulanate'
  ✎ Form: '\bSyp\b' → 'Syrup'

[rx_004] (modified)
  Input:    Cap Omeprazole 20mg OD
  Cleaned:  Capsule Omeprazole 20mg OD
  Expanded: Capsule Omeprazole 20mg once daily
  ✎ Form: '\bCap\b' → 'Capsule'

[rx_005] (modified)
  Inpu

---
## 🤖 Step 4 — Layer 2: Primary NER (Bio-ClinicalBERT + CRF)
Best model for prescription NER per Papers 2, 7, 10 and PRESNER pipeline. Fine-tuned on MIMIC-III clinical notes + n2c2 medication dataset. Identifies: DRUG · DOSAGE · FREQUENCY · FORM.

In [9]:
conda activate nlp_env


CondaError: Run 'conda init' before 'conda activate'


Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install safetensors -U


Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install transformers -U

  Using cached transformers-5.10.2-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.10.2-py3-none-any.whl (11.0 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.1.0
    Uninstalling transformers-5.1.0:
      Successfully uninstalled transformers-5.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gliner 0.2.26 requires transformers<5.2.0,>=4.51.3, but you have transformers 5.10.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [12]:


# ── Load Bio-ClinicalBERT NER model ──────────────────────────────────────
# We use a pre-fine-tuned model from HuggingFace Hub.
# 'samrawal/bert-base-uncased_clinical-ner' is fine-tuned on i2b2/n2c2
# and recognizes: DRUG, STRENGTH, FREQUENCY, ROUTE, DURATION, REASON, ADE

print(f"{Fore.CYAN}Loading Bio-ClinicalBERT NER model...")
print(f"(First run downloads ~440MB — cached after that)\n")

NER_MODEL_NAME = "samrawal/bert-base-uncased_clinical-ner"

try:
    ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
    ner_model     = AutoModelForTokenClassification.from_pretrained(NER_MODEL_NAME)

    # Create the HuggingFace NER pipeline
    # aggregation_strategy='simple' merges subword tokens into full entities
    ner_pipeline = pipeline(
        task                  = "ner",
        model                 = ner_model,
        tokenizer             = ner_tokenizer,
        aggregation_strategy  = "simple",
        device                = 0 if DEVICE.type == "cuda" else -1,
    )
    print(f"{Fore.GREEN}✅ Bio-ClinicalBERT NER loaded")
    print(f"   Model: {NER_MODEL_NAME}")
    print(f"   Labels: {list(ner_model.config.id2label.values())}")

except Exception as e:
    print(f"{Fore.RED}❌ Failed to load primary NER: {e}")
    print(f"{Fore.YELLOW}   → Will use rule-based fallback only")
    ner_pipeline = None


def run_bert_ner(text: str) -> list:
    """
    Run Bio-ClinicalBERT NER on a single text line.
    Returns list of {entity_type, word, score, start, end}
    """
    if ner_pipeline is None:
        return []
    try:
        results = ner_pipeline(text)
        # Normalize entity labels to our 3 target categories
        normalized = []
        for r in results:
            label = r["entity_group"].upper()
            # Map model labels to our categories
            if any(k in label for k in ["DRUG", "MEDICATION", "MED"]):
                r["category"] = "DRUG"
            elif any(k in label for k in ["STRENGTH", "DOSAGE", "DOSE"]):
                r["category"] = "DOSAGE"
            elif any(k in label for k in ["FREQ", "FREQUENCY"]):
                r["category"] = "FREQUENCY"
            elif any(k in label for k in ["FORM", "ROUTE"]):
                r["category"] = "FORM"
            elif any(k in label for k in ["DUR", "DURATION"]):
                r["category"] = "DURATION"
            else:
                r["category"] = label
            normalized.append(r)
        return normalized
    except Exception as e:
        return []


# ── Quick test ────────────────────────────────────────────────────────────
if ner_pipeline:
    test_text = "Tablet Amoxicillin 500mg three times daily"
    test_result = run_bert_ner(test_text)
    print(f"\n{Fore.CYAN}Quick NER test: '{test_text}'")
    for entity in test_result:
        print(f"  [{entity['category']}] '{entity['word']}' (score: {entity['score']:.3f})")

Loading Bio-ClinicalBERT NER model...
(First run downloads ~440MB — cached after that)

❌ Failed to load primary NER: name 'AutoTokenizer' is not defined
   → Will use rule-based fallback only


---
## 🔮 Step 5 — Layer 2B: Fallback NER (GLiNER-BioMed, 2025 SOTA)
Zero-shot biomedical NER — handles drug names unseen during BERT training. Uses natural language labels ('medicine name', 'drug dosage', 'dosage frequency'). From Paper 4's insight on OOV drug names.

In [13]:
from gliner import GLiNER

# ── Load GLiNER-BioMed ────────────────────────────────────────────────────
# Zero-shot: no fine-tuning needed — just describe what you want in English

print(f"{Fore.CYAN}Loading GLiNER-BioMed (zero-shot fallback)...")
print(f"(First run downloads ~350MB — cached after that)\n")

GLINER_MODEL_NAME = "Ihor/gliner-biomed-base-v1.0"

try:
    gliner_model = GLiNER.from_pretrained(GLINER_MODEL_NAME)
    print(f"{Fore.GREEN}✅ GLiNER-BioMed loaded")
    print(f"   Model: {GLINER_MODEL_NAME}")
    print(f"   Mode:  Zero-shot (natural language entity labels)")
except Exception as e:
    print(f"{Fore.RED}❌ Failed to load GLiNER: {e}")
    gliner_model = None

# ── Entity labels we ask GLiNER to find (plain English!) ─────────────────
GLINER_LABELS = [
    "medicine name",
    "drug name",
    "drug dosage amount",
    "dosage frequency",
    "drug form",
]

# Map GLiNER's natural language labels to our categories
GLINER_CATEGORY_MAP = {
    "medicine name":       "DRUG",
    "drug name":           "DRUG",
    "drug dosage amount":  "DOSAGE",
    "dosage frequency":    "FREQUENCY",
    "drug form":           "FORM",
}


def run_gliner_ner(text: str, threshold: float = 0.35) -> list:
    """
    Run GLiNER-BioMed zero-shot NER.
    Lower threshold (0.35) catches more entities — useful for noisy HTR.
    """
    if gliner_model is None:
        return []
    try:
        entities = gliner_model.predict_entities(
            text,
            GLINER_LABELS,
            threshold=threshold
        )
        results = []
        for e in entities:
            results.append({
                "word":          e["text"],
                "category":      GLINER_CATEGORY_MAP.get(e["label"], e["label"].upper()),
                "score":         e["score"],
                "start":         e["start"],
                "end":           e["end"],
                "source":        "gliner",
            })
        return results
    except Exception as e:
        return []


# ── Quick test ────────────────────────────────────────────────────────────
if gliner_model:
    test_text = "Tablet Amoxicillin 500mg three times daily"
    gliner_result = run_gliner_ner(test_text)
    print(f"\n{Fore.CYAN}GLiNER test: '{test_text}'")
    for entity in gliner_result:
        print(f"  [{entity['category']}] '{entity['word']}' (score: {entity['score']:.3f})")

Loading GLiNER-BioMed (zero-shot fallback)...
(First run downloads ~350MB — cached after that)



Fetching 12 files: 100%|██████████| 12/12 [13:20<00:00, 66.71s/it]


✅ GLiNER-BioMed loaded
   Model: Ihor/gliner-biomed-base-v1.0
   Mode:  Zero-shot (natural language entity labels)

GLiNER test: 'Tablet Amoxicillin 500mg three times daily'
  [FORM] 'Tablet' (score: 0.991)
  [DRUG] 'Amoxicillin' (score: 0.938)
  [DOSAGE] '500mg' (score: 0.988)
  [FREQUENCY] 'three times daily' (score: 0.992)


---
## 📏 Step 6 — Layer 3: Rule-Based Parser (Dosage + Frequency)
Based on Paper 10's key finding: BERT excels at frequency phrases, regex/CNN excels at discrete dosage units. This rule layer guarantees we catch structured patterns even when NER misses them.

In [14]:
class PrescriptionRuleParser:
    """
    Layer 3: Rule-based extraction for dosage and frequency.
    Paper 10: Hybrid ensemble pushed results to 0.97 F1 (frequency)
              and 0.95 F1 (dosage) by combining DL + hard regex rules.
    """

    # ── Dosage patterns ───────────────────────────────────────────────────
    DOSAGE_PATTERNS = [
        # e.g. 500mg, 250 mg, 10mg/5ml, 228mg/5ml
        r'\b(\d+(?:\.\d+)?\s*(?:mg|mcg|g|ml|mmol|units?|IU|%))(?:\s*/\s*\d+\s*ml)?\b',
        # Ratio forms: 125mg/5ml
        r'\b(\d+(?:\.\d+)?\s*mg\s*/\s*\d+\s*ml)\b',
    ]

    # ── Frequency patterns ────────────────────────────────────────────────
    FREQUENCY_PATTERNS = [
        # Standard Latin abbreviations
        (r'\bOD\b',           'once daily'),
        (r'\bBD\b',           'twice daily'),
        (r'\bTDS\b',          'three times daily'),
        (r'\bQID\b',          'four times daily'),
        # Numeric shorthand
        (r'\b1\s*[xX]\s*1\b', 'once daily'),
        (r'\b1\s*[xX]\s*2\b', 'twice daily'),
        (r'\b1\s*[xX]\s*3\b', 'three times daily'),
        (r'\b2\s*[xX]\s*3\b', 'three times daily, 2 tablets'),
        (r'\b1\s*[xX]\s*4\b', 'four times daily'),
        # Interval-based
        (r'\bq\s*4\s*h\b',    'every 4 hours'),
        (r'\bq\s*6\s*h\b',    'every 6 hours'),
        (r'\bq\s*8\s*h\b',    'every 8 hours'),
        (r'\bq\s*12\s*h\b',   'every 12 hours'),
        # Timing qualifiers
        (r'\bON\b',           'at night (once nightly)'),
        (r'\bnocte\b',        'at night'),
        (r'\bMane\b',         'in the morning'),
        (r'\bstat\b',         'immediately (single dose)'),
        # Duration qualifier (e.g. x5/7 = for 5 days)
        (r'[xX]\s*(\d+)\s*/\s*7', r'for \1 days'),
    ]

    # ── Drug form patterns ────────────────────────────────────────────────
    FORM_PATTERNS = [
        (r'\b(Tablet|Tab)\b',    'Tablet'),
        (r'\b(Capsule|Cap)\b',   'Capsule'),
        (r'\b(Syrup|Syp)\b',     'Syrup'),
        (r'\b(Injection|Inj)\b', 'Injection'),
        (r'\b(Drops?)\b',        'Drops'),
        (r'\b(Ointment|Oint)\b', 'Ointment'),
        (r'\b(Cream)\b',         'Cream'),
    ]

    def extract(self, text: str) -> dict:
        """Extract dosage, frequency, and form using regex rules."""
        text_upper = text  # preserve case for pattern matching

        # 1. Extract dosages
        dosages = []
        for pattern in self.DOSAGE_PATTERNS:
            matches = re.findall(pattern, text_upper, re.IGNORECASE)
            dosages.extend(matches)
        dosages = list(dict.fromkeys(dosages))  # deduplicate, preserve order

        # 2. Extract frequency
        frequency_raw  = None
        frequency_norm = None
        for pattern, normalized in self.FREQUENCY_PATTERNS:
            match = re.search(pattern, text_upper, re.IGNORECASE)
            if match:
                frequency_raw  = match.group(0)
                # Handle back-reference replacement (e.g. 'for \1 days')
                frequency_norm = re.sub(pattern, normalized, frequency_raw, flags=re.IGNORECASE)
                break

        # 3. Extract drug form
        form = None
        for pattern, normalized in self.FORM_PATTERNS:
            if re.search(pattern, text_upper, re.IGNORECASE):
                form = normalized
                break

        return {
            "dosages":        dosages,
            "dosage":         dosages[0] if dosages else None,     # primary dosage
            "frequency_raw":  frequency_raw,
            "frequency":      frequency_norm,
            "form":           form,
            "rule_confidence": 1.0 if (dosages and frequency_norm) else
                               0.7 if (dosages or  frequency_norm) else 0.3,
        }


# ── Test the rule parser ──────────────────────────────────────────────────
rule_parser = PrescriptionRuleParser()

print(f"{Fore.CYAN}{'─'*60}")
print(f"{Fore.CYAN}Layer 3: Rule-Based Parser Results")
print(f"{Fore.CYAN}{'─'*60}")

for item in SAMPLE_HTR_OUTPUTS[:8]:
    result = rule_parser.extract(item["line"])
    print(f"\n[{item['id']}] {item['line']}")
    print(f"  Dosage:    {result['dosage']}")
    print(f"  Frequency: {result['frequency']}  (raw: {result['frequency_raw']})")
    print(f"  Form:      {result['form']}")
    print(f"  Confidence:{result['rule_confidence']:.1f}")

────────────────────────────────────────────────────────────
Layer 3: Rule-Based Parser Results
────────────────────────────────────────────────────────────

[rx_001] Tab Amoxicillin 500mg 1x3
  Dosage:    500mg
  Frequency: three times daily  (raw: 1x3)
  Form:      Tablet
  Confidence:1.0

[rx_002] Tab Paracetamol 500 mg TDS
  Dosage:    500 mg
  Frequency: three times daily  (raw: TDS)
  Form:      Tablet
  Confidence:1.0

[rx_003] Syp Augmentin 228mg/5ml BD
  Dosage:    228mg
  Frequency: twice daily  (raw: BD)
  Form:      Syrup
  Confidence:1.0

[rx_004] Cap Omeprazole 20mg OD
  Dosage:    20mg
  Frequency: once daily  (raw: OD)
  Form:      Capsule
  Confidence:1.0

[rx_005] Tab Amoxl 500rng 1xd
  Dosage:    None
  Frequency: None  (raw: None)
  Form:      Tablet
  Confidence:0.3

[rx_006] Tab Metroniclazol 400mg BD
  Dosage:    400mg
  Frequency: twice daily  (raw: BD)
  Form:      Tablet
  Confidence:1.0

[rx_007] Cap 0meprazole 2Omg OD
  Dosage:    None
  Frequency: once dail

---
## 🔍 Step 7 — Layer 4: Fuzzy Normalization (RapidFuzz + RxNorm)
Corrects HTR spelling errors in drug names by matching against a known drug dictionary. Uses RapidFuzz locally + RxNorm API as authoritative fallback. From Papers 9, 10 and the harmonization research.

In [15]:
from rapidfuzz import fuzz, process
import requests

class DrugNameNormalizer:
    """
    Layer 4: Fuzzy matching drug names against known dictionaries.
    Strategy (cascaded):
      1. Exact match in local dictionary
      2. Fuzzy match (RapidFuzz) with score >= 80
      3. RxNorm API approximate match (handles brand names, generics)
      4. Flag as needs_review if no match found
    """

    # ── Local drug dictionary — common prescription drugs ─────────────────
    # Expand this list with drugs common in your region
    LOCAL_DRUG_DICT = [
        # Antibiotics
        "Amoxicillin", "Amoxicillin Clavulanate", "Augmentin",
        "Ampicillin", "Cloxacillin", "Flucloxacillin",
        "Metronidazole", "Tinidazole",
        "Ciprofloxacin", "Levofloxacin",
        "Erythromycin", "Azithromycin", "Clarithromycin",
        "Doxycycline", "Tetracycline",
        "Cefalexin", "Cefuroxime", "Ceftriaxone",
        "Gentamicin", "Gentamycin",
        # Analgesics / Antipyretics
        "Paracetamol", "Acetaminophen",
        "Ibuprofen", "Diclofenac", "Naproxen",
        "Aspirin", "Mefenamic Acid",
        "Tramadol", "Codeine", "Morphine",
        # Gastrointestinal
        "Omeprazole", "Lansoprazole", "Pantoprazole",
        "Ranitidine", "Famotidine",
        "Domperidone", "Metoclopramide",
        "Lactulose", "Bisacodyl",
        # Cardiovascular
        "Atorvastatin", "Simvastatin", "Rosuvastatin",
        "Amlodipine", "Nifedipine",
        "Enalapril", "Lisinopril", "Ramipril",
        "Metoprolol", "Atenolol", "Bisoprolol",
        "Warfarin", "Aspirin",
        # Diabetes
        "Metformin", "Glibenclamide", "Glipizide",
        "Insulin",
        # Respiratory
        "Salbutamol", "Albuterol", "Ipratropium",
        "Prednisolone", "Dexamethasone",
        "Cetirizine", "Loratadine", "Chlorpheniramine",
        # Others
        "Diazepam", "Clonazepam",
        "Amitriptyline", "Sertraline",
        "Folic Acid", "Ferrous Sulphate",
        "Vitamin C", "Vitamin D",
    ]

    # Pre-process dict to lowercase for matching
    def __init__(self):
        self.dict_lower = {d.lower(): d for d in self.LOCAL_DRUG_DICT}

    def normalize(self, drug_name: str, fuzzy_threshold: int = 78) -> dict:
        """
        Normalize a drug name through cascaded matching.
        Returns: {normalized_name, method, score, needs_review}
        """
        if not drug_name or len(drug_name.strip()) < 2:
            return {"normalized_name": drug_name, "method": "none",
                    "score": 0.0, "needs_review": True}

        query = drug_name.strip()

        # ── Strategy 1: Exact match ───────────────────────────────────────
        if query.lower() in self.dict_lower:
            return {
                "normalized_name": self.dict_lower[query.lower()],
                "method":          "exact",
                "score":           1.0,
                "needs_review":    False,
            }

        # ── Strategy 2: RapidFuzz local matching ─────────────────────────
        best_match, best_score, _ = process.extractOne(
            query.lower(),
            self.dict_lower.keys(),
            scorer=fuzz.token_sort_ratio,
        )
        if best_score >= fuzzy_threshold:
            return {
                "normalized_name": self.dict_lower[best_match],
                "method":          f"fuzzy (score={best_score})",
                "score":           best_score / 100,
                "needs_review":    best_score < 85,  # flag borderline matches
            }

        # ── Strategy 3: RxNorm API (free, no API key needed) ─────────────
        rxnorm_result = self._query_rxnorm(query)
        if rxnorm_result:
            return rxnorm_result

        # ── Strategy 4: No match — flag for review ────────────────────────
        return {
            "normalized_name": query,  # keep original
            "method":          "unmatched",
            "score":           0.0,
            "needs_review":    True,
        }

    def _query_rxnorm(self, drug_name: str) -> dict | None:
        """Query RxNorm API for approximate drug name matching."""
        try:
            # RxNorm approximate term search — handles misspellings
            url = "https://rxnav.nlm.nih.gov/REST/approximateTerm.json"
            resp = requests.get(
                url,
                params={"term": drug_name, "maxEntries": 1},
                timeout=5
            )
            if resp.status_code != 200:
                return None

            data = resp.json()
            candidates = data.get("approximateGroup", {}).get("candidate", [])
            if not candidates:
                return None

            top = candidates[0]
            score = int(top.get("score", 0))
            name  = top.get("name", drug_name)

            if score >= 70:
                return {
                    "normalized_name": name,
                    "method":          f"rxnorm (score={score})",
                    "score":           score / 100,
                    "needs_review":    score < 85,
                    "rxcui":           top.get("rxcui"),
                }
        except Exception:
            pass  # RxNorm offline or timeout — continue
        return None


# ── Test the normalizer ───────────────────────────────────────────────────
normalizer = DrugNameNormalizer()

print(f"{Fore.CYAN}{'─'*60}")
print(f"{Fore.CYAN}Layer 4: Fuzzy Normalization Tests")
print(f"{Fore.CYAN}{'─'*60}")

test_drugs = [
    "Amoxl",          # HTR noise
    "Metroniclazol",  # misspelled
    "0meprazole",     # OCR artifact
    "lbuprofen",      # l vs I
    "Atorvastatln",   # ln vs in
    "Amoxicilin",     # one l missing
    "Cetirizne",      # missing i
    "Amoxicillin",    # correct — exact match
    "UnknownDrug123", # no match
]

for drug in test_drugs:
    result = normalizer.normalize(drug)
    flag  = f" {Fore.YELLOW}[REVIEW]" if result["needs_review"] else ""
    color = Fore.GREEN if result["method"] == "exact" else \
            Fore.CYAN  if "fuzzy"  in result["method"] else \
            Fore.BLUE  if "rxnorm" in result["method"] else Fore.RED
    print(f"  {color}'{drug}' → '{result['normalized_name']}'"
          f" [{result['method']}]{flag}")

────────────────────────────────────────────────────────────
Layer 4: Fuzzy Normalization Tests
────────────────────────────────────────────────────────────
  'Amoxl' → 'Amoxl' [unmatched] [REVIEW]
  'Metroniclazol' → 'Metronidazole' [fuzzy (score=84.61538461538461)] [REVIEW]
  '0meprazole' → 'Omeprazole' [fuzzy (score=90.0)]
  'lbuprofen' → 'Ibuprofen' [fuzzy (score=88.88888888888889)]
  'Atorvastatln' → 'Atorvastatin' [fuzzy (score=91.66666666666666)]
  'Amoxicilin' → 'Amoxicillin' [fuzzy (score=95.23809523809523)]
  'Cetirizne' → 'Cetirizine' [fuzzy (score=94.73684210526316)]
  'Amoxicillin' → 'Amoxicillin' [exact]
  'UnknownDrug123' → 'UnknownDrug123' [unmatched] [REVIEW]


---
## ⚡ Step 8 — Full Pipeline: Combine All Layers
The complete extraction pipeline — pre-process → BERT NER → GLiNER fallback → rule parser → fuzzy normalization → confidence gate.

In [17]:
from tqdm import tqdm            # plain text progress bar, no widgets needed

In [18]:
CONFIDENCE_THRESHOLD = 0.85  # below this → flag for review


def extract_prescription_entities(htr_item: dict) -> dict:
    """
    Full 5-layer NLP pipeline for a single HTR line.
    """
    rx_id   = htr_item["id"]
    raw_text = htr_item["line"]

    # ── Layer 1: Pre-processing ───────────────────────────────────────────
    pre = preprocessor.clean(raw_text)
    text_for_ner  = pre["expanded"]   # frequency abbrevs expanded
    text_for_rule = pre["cleaned"]    # abbreviations NOT expanded (rule parser handles)

    # ── Layer 2: Bio-ClinicalBERT NER ────────────────────────────────────
    bert_entities = run_bert_ner(text_for_ner)

    # ── Layer 2B: GLiNER fallback (if BERT found no DRUG entity) ─────────
    bert_drugs = [e for e in bert_entities if e["category"] == "DRUG"]
    if not bert_drugs:
        gliner_entities = run_gliner_ner(text_for_ner)
        # Merge GLiNER results, mark source
        for e in gliner_entities:
            e["source"] = "gliner"
        combined_entities = bert_entities + gliner_entities
    else:
        for e in bert_entities:
            e["source"] = "bert"
        combined_entities = bert_entities

    # ── Layer 3: Rule-based parser ────────────────────────────────────────
    rule_result = rule_parser.extract(text_for_rule)

    # ── Extract primary entities (merge NER + rules) ──────────────────────
    # Drug name: from NER (BERT preferred, GLiNER fallback)
    drug_entities = [e for e in combined_entities if e["category"] == "DRUG"]
    drug_entity   = max(drug_entities, key=lambda x: x["score"]) if drug_entities else None
    drug_raw      = drug_entity["word"] if drug_entity else None
    drug_score    = drug_entity["score"] if drug_entity else 0.0

    # Dosage: NER preferred, rule parser as fallback
    dosage_entities = [e for e in combined_entities if e["category"] == "DOSAGE"]
    if dosage_entities:
        dosage_raw   = max(dosage_entities, key=lambda x: x["score"])["word"]
        dosage_score = max(dosage_entities, key=lambda x: x["score"])["score"]
    elif rule_result["dosage"]:
        dosage_raw   = rule_result["dosage"]
        dosage_score = rule_result["rule_confidence"]
    else:
        dosage_raw, dosage_score = None, 0.0

    # Frequency: NER preferred, rule parser as fallback
    freq_entities = [e for e in combined_entities if e["category"] == "FREQUENCY"]
    if freq_entities:
        frequency_raw   = max(freq_entities, key=lambda x: x["score"])["word"]
        frequency_score = max(freq_entities, key=lambda x: x["score"])["score"]
    elif rule_result["frequency"]:
        frequency_raw   = rule_result["frequency"]
        frequency_score = rule_result["rule_confidence"]
    else:
        frequency_raw, frequency_score = None, 0.0

    # Form
    form_entities = [e for e in combined_entities if e["category"] == "FORM"]
    form = form_entities[0]["word"] if form_entities else rule_result.get("form")

    # ── Layer 4: Fuzzy normalization of drug name ─────────────────────────
    if drug_raw:
        norm_result   = normalizer.normalize(drug_raw)
        medicine_name = norm_result["normalized_name"]
        norm_method   = norm_result["method"]
        norm_review   = norm_result["needs_review"]
    else:
        medicine_name = None
        norm_method   = "no_drug_found"
        norm_review   = True

    # ── Layer 5: Confidence gate ──────────────────────────────────────────
    # Weighted composite score
    scores = [s for s in [drug_score, dosage_score * 0.5, frequency_score * 0.5] if s > 0]
    overall_confidence = sum(scores) / len(scores) if scores else 0.0

    needs_review = (
        overall_confidence < CONFIDENCE_THRESHOLD or
        medicine_name is None or
        norm_review
    )

    # ── Build final structured output ─────────────────────────────────────
    return {
        "id":                rx_id,
        "raw_text":          raw_text,
        "preprocessed_text": pre["cleaned"],
        "medicine_name":     medicine_name,
        "medicine_raw":      drug_raw,
        "dosage":            dosage_raw,
        "frequency":         frequency_raw,
        "form":              form,
        "confidence":        round(overall_confidence, 3),
        "needs_review":      needs_review,
        "normalization_method": norm_method,
        "preprocessing_changes": pre["changes"],
        "all_entities":      combined_entities,
    }


# ── Run the full pipeline on all HTR lines ────────────────────────────────
print(f"{Fore.CYAN}{'─'*60}")
print(f"{Fore.CYAN}Running Full NLP Pipeline on {len(SAMPLE_HTR_OUTPUTS)} Lines")
print(f"{Fore.CYAN}{'─'*60}\n")

all_results = []
for item in tqdm(SAMPLE_HTR_OUTPUTS, desc="Processing lines"):
    result = extract_prescription_entities(item)
    all_results.append(result)

print(f"\n{Fore.GREEN}✅ Pipeline complete!")
print(f"   Total lines:     {len(all_results)}")
print(f"   Needs review:    {sum(r['needs_review'] for r in all_results)}")
print(f"   Auto-confirmed:  {sum(not r['needs_review'] for r in all_results)}")

────────────────────────────────────────────────────────────
Running Full NLP Pipeline on 15 Lines
────────────────────────────────────────────────────────────




Processing lines: 100%|██████████| 15/15 [00:02<00:00,  6.12it/s]



✅ Pipeline complete!
   Total lines:     15
   Needs review:    15
   Auto-confirmed:  0


---
## 📊 Step 9 — Results Display & Analysis

In [19]:
print(f"\n{Fore.CYAN}{'═'*70}")
print(f"{Fore.CYAN}  PRESCRIPTION NLP EXTRACTION RESULTS")
print(f"{Fore.CYAN}{'═'*70}")

for r in all_results:
    status_color = Fore.RED    if r["needs_review"] else \
                   Fore.YELLOW if r["confidence"] < 0.92 else Fore.GREEN
    flag = " ⚠️  [REVIEW]" if r["needs_review"] else " ✅"

    print(f"\n{Fore.WHITE}[{r['id']}] {r['raw_text']}")
    if r["preprocessed_text"] != r["raw_text"]:
        print(f"  {Fore.YELLOW}Cleaned: {r['preprocessed_text']}")
    print(f"  {Fore.CYAN}Medicine:  {r['medicine_name']}  (raw: '{r['medicine_raw']}', via: {r['normalization_method']})")
    print(f"  {Fore.CYAN}Dosage:    {r['dosage']}")
    print(f"  {Fore.CYAN}Frequency: {r['frequency']}")
    print(f"  {Fore.CYAN}Form:      {r['form']}")
    print(f"  {status_color}Confidence: {r['confidence']:.3f}{flag}")

# ── Summary DataFrame ─────────────────────────────────────────────────────
print(f"\n\n{Fore.CYAN}{'─'*60}")
print(f"{Fore.CYAN}SUMMARY TABLE")
print(f"{Fore.CYAN}{'─'*60}")

df = pd.DataFrame([{
    "ID":           r["id"],
    "Medicine":     r["medicine_name"],
    "Dosage":       r["dosage"],
    "Frequency":    r["frequency"],
    "Confidence":   r["confidence"],
    "Review":       "⚠️" if r["needs_review"] else "✅",
} for r in all_results])

print(df.to_string(index=False))


══════════════════════════════════════════════════════════════════════
  PRESCRIPTION NLP EXTRACTION RESULTS
══════════════════════════════════════════════════════════════════════

[rx_001] Tab Amoxicillin 500mg 1x3
  Cleaned: Tablet Amoxicillin 500mg 1x3
  Medicine:  Amoxicillin  (raw: 'Amoxicillin', via: exact)
  Dosage:    500mg
  Frequency: three times daily
  Form:      Tablet
  Confidence: 0.643 ⚠️  [REVIEW]

[rx_002] Tab Paracetamol 500 mg TDS
  Cleaned: Tablet Paracetamol 500 mg TDS
  Medicine:  Paracetamol  (raw: 'Paracetamol', via: exact)
  Dosage:    500 mg
  Frequency: three times daily
  Form:      Tablet
  Confidence: 0.617 ⚠️  [REVIEW]

[rx_003] Syp Augmentin 228mg/5ml BD
  Cleaned: Syrup Amoxicillin Clavulanate 228mg/5ml BD
  Medicine:  Amoxicillin Clavulanate  (raw: 'Amoxicillin Clavulanate', via: exact)
  Dosage:    228mg/5ml
  Frequency: twice daily
  Form:      Syrup
  Confidence: 0.608 ⚠️  [REVIEW]

[rx_004] Cap Omeprazole 20mg OD
  Cleaned: Capsule Omeprazole 20m

---
## 💾 Step 10 — Save All Outputs

In [20]:
# ── 1. Save full results JSON ─────────────────────────────────────────────
results_path = os.path.join(NLP_OUTPUT_DIR, "results.json")
# Serialize — remove torch tensors from entity list
serializable_results = []
for r in all_results:
    r_clean = {k: v for k, v in r.items() if k != "all_entities"}
    r_clean["entities"] = [
        {
            "word":     e.get("word", ""),
            "category": e.get("category", ""),
            "score":    float(e.get("score", 0)),
            "source":   e.get("source", ""),
        }
        for e in r.get("all_entities", [])
    ]
    serializable_results.append(r_clean)

with open(results_path, "w") as f:
    json.dump(serializable_results, f, indent=2, default=str)
print(f"{Fore.GREEN}✅ results.json saved → {results_path}")

# ── 2. Save flagged items for review ─────────────────────────────────────
flagged = [r for r in serializable_results if r["needs_review"]]
flagged_path = os.path.join(NLP_OUTPUT_DIR, "flagged_review.json")
with open(flagged_path, "w") as f:
    json.dump(flagged, f, indent=2, default=str)
print(f"{Fore.GREEN}✅ flagged_review.json saved → {flagged_path}")
print(f"   {len(flagged)} items need manual review")

# ── 3. Save pipeline objects for production reuse ─────────────────────────
pipeline_objects = {
    "preprocessor":  preprocessor,
    "rule_parser":   rule_parser,
    "normalizer":    normalizer,
}
pipeline_path = os.path.join(NLP_OUTPUT_DIR, "nlp_pipeline.pkl")
with open(pipeline_path, "wb") as f:
    pickle.dump(pipeline_objects, f)
print(f"{Fore.GREEN}✅ nlp_pipeline.pkl saved → {pipeline_path}")

# ── 4. Update pipeline_config.json with NLP stage info ───────────────────
pipeline_config["nlp_stage"] = {
    "status":           "complete",
    "model_primary":    "Bio-ClinicalBERT + CRF (samrawal/bert-base-uncased_clinical-ner)",
    "model_fallback":   "GLiNER-BioMed (Ihor/gliner-biomed-base-v1.0)",
    "normalization":    "RapidFuzz + RxNorm API",
    "entities_extracted": ["medicine_name", "dosage", "frequency", "form"],
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "output_files": {
        "results":  "nlp_output/results.json",
        "flagged":  "nlp_output/flagged_review.json",
        "pipeline": "nlp_output/nlp_pipeline.pkl",
    },
}
with open(CONFIG_PATH, "w") as f:
    json.dump(pipeline_config, f, indent=2)
print(f"{Fore.GREEN}✅ pipeline_config.json updated")

print(f"\n{Fore.CYAN}{'═'*55}")
print(f"{Fore.GREEN}  NLP PIPELINE COMPLETE")
print(f"{Fore.CYAN}{'═'*55}")
print(f"""
Output folder: {NLP_OUTPUT_DIR}
  ├── results.json          ({len(serializable_results)} prescriptions structured)
  ├── flagged_review.json   ({len(flagged)} items need review)
  └── nlp_pipeline.pkl      (reusable pipeline objects)
""")

✅ results.json saved → /Users/avishkashenan/Desktop/Line Segmentation New/htr_input/nlp_output/results.json
✅ flagged_review.json saved → /Users/avishkashenan/Desktop/Line Segmentation New/htr_input/nlp_output/flagged_review.json
   15 items need manual review
✅ nlp_pipeline.pkl saved → /Users/avishkashenan/Desktop/Line Segmentation New/htr_input/nlp_output/nlp_pipeline.pkl
✅ pipeline_config.json updated

═══════════════════════════════════════════════════════
  NLP PIPELINE COMPLETE
═══════════════════════════════════════════════════════

Output folder: /Users/avishkashenan/Desktop/Line Segmentation New/htr_input/nlp_output
  ├── results.json          (15 prescriptions structured)
  ├── flagged_review.json   (15 items need review)
  └── nlp_pipeline.pkl      (reusable pipeline objects)



---
## 🚀 Step 11 — Production Usage
How to plug this into your existing pipeline when your HTR model is ready.

In [21]:
# ── Production single-line usage ──────────────────────────────────────────
# Once your HTR model outputs text, call this function:

def process_single_prescription_line(line_text: str, line_id: str = "rx") -> dict:
    """
    Process a single line of HTR output through the full NLP pipeline.

    Args:
        line_text: raw text from your HTR model (e.g. 'Tab Amoxl 500mg 1x3')
        line_id:   identifier for this line (e.g. filename without extension)

    Returns:
        dict with: medicine_name, dosage, frequency, form, confidence, needs_review
    """
    return extract_prescription_entities({"id": line_id, "line": line_text})


# ── Demo ──────────────────────────────────────────────────────────────────
print(f"{Fore.CYAN}Production usage demo:\n")

demo_lines = [
    "Tab Amoxl 500rng 1xd",
    "Cap Omeprazole 20mg OD",
    "Syp Augmentin 228mg/5ml BD",
]

for line in demo_lines:
    result = process_single_prescription_line(line)
    print(f"Input:   {line}")
    print(f"Output:  medicine={result['medicine_name']} | "
          f"dosage={result['dosage']} | "
          f"frequency={result['frequency']} | "
          f"confidence={result['confidence']:.2f}")
    print()

# ── How to connect to your HTR model ─────────────────────────────────────
print(f"{Fore.CYAN}{'─'*55}")
print(f"{Fore.CYAN}Connecting to your HTR model (replace this block):")
print(f"""
# In your HTR pipeline script:
import pickle, json

# 1. Load saved NLP pipeline
with open('htr_input/nlp_output/nlp_pipeline.pkl', 'rb') as f:
    nlp = pickle.load(f)

# 2. For each line your HTR model processes:
for img_file in sorted(os.listdir(LINE_CROPS_DIR)):
    htr_text = your_htr_model.predict(img_file)   # ← your HTR here
    nlp_result = process_single_prescription_line(htr_text, img_file)
    print(nlp_result)  # → structured extraction
""")

Production usage demo:

Input:   Tab Amoxl 500rng 1xd
Output:  medicine=Amoxl | dosage=500mg | frequency=once daily | confidence=0.59

Input:   Cap Omeprazole 20mg OD
Output:  medicine=Omeprazole | dosage=20mg | frequency=once daily | confidence=0.64

Input:   Syp Augmentin 228mg/5ml BD
Output:  medicine=Amoxicillin Clavulanate | dosage=228mg/5ml | frequency=twice daily | confidence=0.61

───────────────────────────────────────────────────────
Connecting to your HTR model (replace this block):

# In your HTR pipeline script:
import pickle, json

# 1. Load saved NLP pipeline
with open('htr_input/nlp_output/nlp_pipeline.pkl', 'rb') as f:
    nlp = pickle.load(f)

# 2. For each line your HTR model processes:
for img_file in sorted(os.listdir(LINE_CROPS_DIR)):
    htr_text = your_htr_model.predict(img_file)   # ← your HTR here
    nlp_result = process_single_prescription_line(htr_text, img_file)
    print(nlp_result)  # → structured extraction



---
## 🔬 Step 12 — Evaluation & Metrics (Optional)
Once you have ground-truth labels, use this to measure accuracy.

In [22]:
# ── Ground truth format — fill in when you have labeled data ─────────────
GROUND_TRUTH = [
    # {"id": "rx_001", "medicine_name": "Amoxicillin", "dosage": "500mg", "frequency": "three times daily"},
    # Add your labeled examples here
]

if not GROUND_TRUTH:
    print(f"{Fore.YELLOW}⚠️  No ground truth provided yet.")
    print(f"   Add labeled examples to GROUND_TRUTH to measure accuracy.")
    print(f"""\n   Format:
   GROUND_TRUTH = [
       {{\"id\": \"rx_001\",
        \"medicine_name\": \"Amoxicillin\",
        \"dosage\": \"500mg\",
        \"frequency\": \"three times daily\"}},
       ...
   ]""")
else:
    # Compute exact-match accuracy per entity type
    gt_dict = {g["id"]: g for g in GROUND_TRUTH}
    metrics  = {"medicine": [], "dosage": [], "frequency": []}

    for r in all_results:
        if r["id"] not in gt_dict:
            continue
        gt = gt_dict[r["id"]]
        metrics["medicine"].append(
            r["medicine_name"].lower() == gt["medicine_name"].lower() if r["medicine_name"] else False
        )
        metrics["dosage"].append(
            r["dosage"] == gt["dosage"] if r["dosage"] else False
        )
        metrics["frequency"].append(
            r["frequency"].lower() == gt["frequency"].lower() if r["frequency"] else False
        )

    print(f"\n{Fore.CYAN}Evaluation Results (exact match):")
    for entity, scores in metrics.items():
        acc = sum(scores) / len(scores) * 100 if scores else 0
        print(f"  {entity.capitalize()}: {acc:.1f}% ({sum(scores)}/{len(scores)})")

⚠️  No ground truth provided yet.
   Add labeled examples to GROUND_TRUTH to measure accuracy.

   Format:
   GROUND_TRUTH = [
       {"id": "rx_001",
        "medicine_name": "Amoxicillin",
        "dosage": "500mg",
        "frequency": "three times daily"},
       ...
   ]
